In [1]:
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from agent_graph import StagePlayWriter,input_message, StagePlayState
from langgraph.types import Command
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.checkpoint.sqlite import SqliteSaver
import time

from uuid import uuid4

### Setup agent graph

In [9]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
# tools = [get_character_description, create_character, human_assistance]

# thread_id = uuid4()
thread_id = "this_thread_2"

graph_config: RunnableConfig = RunnableConfig({"configurable": {"thread_id": thread_id}})

playwriter = StagePlayWriter(
    llm=llm,
    themes= """Loss of innocence, Becoming Psychologically whole, Jungian Psychology, Bildung""",
    vibe= """Subtly, Weird and funky""",
    setting= "Tam Tamoree, fictional town in German Bavaria",
    number_of_chapters= 4
)
conn_checkptr = "db/graph_checkpoints/checkpoints.db"


### Graph invoke and continue functions

In [3]:
async def start_graph(init_message: str ,graph_config:  RunnableConfig) -> dict[str, str]:
    """Start the agent application
    """
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=input_message(init_message), config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()

    return {"status": "Graph started, may be paused"}


async def resume_graph(input: str, graph_config) -> dict[str, str]:
    """Resume graph after break from human in the loop tool call
    """
    resume_input = Command(resume= {"data": input}) 
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=resume_input, config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()
    return {"status": "Resumed, may stopp again"}



In [4]:
def run_graph(message: str, graph_config:  RunnableConfig) -> dict[str, str]:
    """Start the agent application
    """
    with SqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        for event in (playwriter
                        .build_graph(checkpointer)
                        .stream(input=input_message(message), # type: ignore
                                config= graph_config, 
                                stream_mode="values")): 

            context = event["context"][-1] 
            print(context)
    return {"status": "Graph started, may be paused"}


In [ ]:
with SqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
    graph = playwriter.build_graph(checkpointer)

    output = graph.invoke(input=input_message("""Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        """), # type: ignore
                                config= graph_config, 
                                stream_mode="values")

In [5]:
with SqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
    graph = playwriter.build_graph(checkpointer)
    
    for i,message in enumerate(graph.get_state(config= graph_config)[0]['context']):
        print(i, message.content)
    

0 Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        
1 Got it! Please provide the preceding text or context and the role for which I need to write the next line.
2 Got it! Please provide me with the next line or context for the play, and I will write the continuation accordingly.
3 Narrator: The sun began to set on the crooked streets of Tam Tamoree, casting elongated shadows that danced with whispers of the past. Luna stood at the edge of the town square, arms crossed and a defiant gaze fixed upon the horizon, where the clouds gathered for a storm. The smell of rain mingled with the scent of the fried pastries from the vendor, igniting a bittersweet yearning within her.

Luna: "You know, it might rain, but that won't stop me from making my mark on this town! I've got plans, Swedenborg. Big ones!"
4 Sure! Please provide me with the current state of

In [21]:
print(playwriter.system_message(line_count=300, synopsis= playwriter.current_synopsis).content)


    You are the co writer of a stage play
    You recieve a story and write the continuation
    As you wil see below, you recieve a header of information about the play, and its current state.
    This information is structured to include: 
     1. The general features of the play such as the themes, vibe, and setting
     2. Scene and Arc description: A list of scenes and the general direction the story should
     move towards in each scene. 
     3. The roster for the current scene, with a small character description for each character.
     4. If not at first chapter you will recieve a small synopsis of the preceding chapters. 
 
    You will recieve the current state of the play with a precursor to a new line for a given role.
    This role will either be the role of the narrator or a character in the play.
    When writing as the narrator you will continue the story by describing the
    unfolding of events based on previous context. When writing for a character you 
    will w

In [ ]:
playwriter.current_synopsis

['Certainly! Here’s a synopsis of the provided lines:\n\n---\n\nIn a tense moment, Luna questions the widespread fear surrounding Rory, suggesting he is merely an ordinary person. Captain Flask warns her that Rory\'s presence can be as turbulent and uncontrollable as a storm, hinting at the darker aspects of Rory\'s character that others perceive. Luna\'s response reveals her increasing distress, dramatically stating that she feels as if her blood is infested with "ants," symbolizing her internal turmoil and fear. She pleads for help, overwhelmed by the imagined creatures within her, which may reflect deeper psychological issues or existential dread.\n\n---\n\nThis synopsis encapsulates the emotional and thematic elements of the dialogue, highlighting the characters\' fears and the ominous undertones present in their interactions. If you have any specific elements you\'d like to expand on or modify, let me know!']

### Call graph 

In [18]:
await start_graph(init_message= """Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        """, graph_config= graph_config)

================================ Human Message =================================

Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        
================================== Ai Message ==================================
Tool Calls:
  create_character (call_jZbH4k9FgopMEMsyXrpTQPha)
 Call ID: call_jZbH4k9FgopMEMsyXrpTQPha
  Args:
    character_name: Anya
    age: 50
    gender: F
    disposition: Mysterious
    relationships_in: {'Townspeople': 'Eccentric local', 'Luna': 'Guiding mentor'}
  create_character (call_p7LIYZsBmMX3yRGXEYbiV773)
 Call ID: call_p7LIYZsBmMX3yRGXEYbiV773
  Args:
    character_name: Max
    age: 19
    gender: M
    disposition: Reckless
    relationships_in: {'Townspeople': 'Aloof', 'Rory': 'Frenemy'}
  create_character (call_X9STEDPuP2CcngWBGNzYnHhY)
 Call ID: call_X9STEDPuP2CcngWBGNzYnHhY
  Args:
    character_name: Franzi
    ag

{'status': 'Graph started, may be paused'}

In [14]:
await resume_graph("""Luna:
                   Oh my god! My blood is ants. I am dying Swedenborg. Oh! The pain!! I can't see. The creepy crawlies, they're within me. Help! """, 
                   graph_config= graph_config)

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_3sdylspYBQ92rZlu1JZ7ROjh)
 Call ID: call_3sdylspYBQ92rZlu1JZ7ROjh
  Args:
    query: Please provide the next line for Captain Flask after the following context: 

Luna: Why does everyone act like they’re so afraid of Rory? He’s just a guy, right? 

Captain Flask: But Luna, sometimes a guy like Rory can be a storm you can’t escape from.
================================= Tool Message =================================
Name: human_assistance

Luna:
                   Oh my god! My blood is ants. I am dying Swedenborg. Oh! The pain!! I can't see. The creepy crawlies, they're within me. Help! 
================================== Ai Message ==================================

It seems we have reached the end of the current chapter with 15 lines already set. As we prepare to transition into the next chapter of the play, I suggest we can introduce a pivotal moment that propels th

{'status': 'Resumed, may stopp again'}

In [16]:
await resume_graph("""Swedenborg:
                   Whatever! You're being weird today """, graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}

In [17]:
await resume_graph("""Swedenborg:
                   Whatever! You're being weird today """, graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}